# Sunday Work: Minimal Single-Study Baseline
Goal: use IgLM heavy-chain embeddings, then train a small MLP on the **largest study only** to check whether validation behavior improves.

## 1) Setup

In [ ]:
from __future__ import annotations

import json
import math
import os
from dataclasses import dataclass
from pathlib import Path

# Avoid slow matplotlib font-cache rebuild in unwritable home dir.
os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".mplconfig").resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import transformers

from iglm.model.IgLM import CHECKPOINT_DICT, VOCAB_FILE

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / 'PDW_GeneralAntibodies').exists():
        return cwd / 'PDW_GeneralAntibodies'
    if cwd.name == 'PDW_GeneralAntibodies':
        return cwd
    for parent in [cwd, *cwd.parents]:
        cand = parent / 'PDW_GeneralAntibodies'
        if cand.exists():
            return cand
    raise FileNotFoundError('Could not locate PDW_GeneralAntibodies from current working directory.')


ROOT = resolve_project_root()

@dataclass
class CFG:
    data_csv: Path = ROOT / 'flab_thermo_unified_ml_tm_only.csv'
    cache_npz: Path = ROOT / 'sunday_work' / 'iglm_heavy_cache.npz'
    cache_map: Path = ROOT / 'sunday_work' / 'iglm_heavy_cache_map.json'
    recompute_embeddings: bool = False

    model_name: str = 'IgLM'
    species_token: str = '[HUMAN]'

    train_frac: float = 0.70
    val_frac: float = 0.15
    test_frac: float = 0.15
    seed: int = 42

    hidden_dim: int = 64
    dropout: float = 0.30
    lr: float = 1e-3
    weight_decay: float = 2e-3
    batch_size: int = 64
    max_epochs: int = 250
    patience: int = 25

cfg = CFG()
set_seed(cfg.seed)
print('Project root:', ROOT)
print('Using CSV:', cfg.data_csv)
print(cfg)


## 2) Load data and keep only the largest experiment

In [ ]:
df = pd.read_csv(cfg.data_csv)
df['heavy'] = df['heavy'].fillna('').astype(str).str.strip().str.upper()
df['heavy'] = df['heavy'].replace({'NAN':'', 'NONE':'', 'NULL':''})
df['y'] = pd.to_numeric(df['y'], errors='coerce')
df = df[(df['heavy'] != '') & df['y'].notna()].reset_index(drop=True)

counts = df['source_file'].value_counts()
largest_study = counts.index[0]
print('Largest study:', largest_study, 'n=', int(counts.iloc[0]))

study_df = df[df['source_file'].eq(largest_study)].reset_index(drop=True)
print('Rows kept:', len(study_df))
study_df.head()


## 3) Embed heavy chains with IgLM (cached)

In [ ]:
def load_vocab() -> dict[str, int]:
    tok2id = {}
    with open(VOCAB_FILE, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            tok = line.strip()
            if tok:
                tok2id[tok] = i
    return tok2id


def embed_heavy(model, device, tok2id, seq: str, species_token: str) -> np.ndarray:
    tokens = ['[HEAVY]', species_token] + list(seq) + ['[SEP]']
    unk_id = tok2id.get('[UNK]', 1)
    ids = [tok2id.get(t, unk_id) for t in tokens]
    if any(i == unk_id for i in ids):
        raise ValueError('Unknown token')
    x = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        out = model(x, output_hidden_states=True, return_dict=True)
    hs = out.hidden_states[-1][0]
    aa_hs = hs[2:-1]
    if aa_hs.shape[0] == 0:
        raise ValueError('Empty sequence after token trimming')
    return aa_hs.mean(dim=0).cpu().numpy().astype(np.float32)


def save_cache(seq2emb: dict[str, np.ndarray], cfg: CFG) -> None:
    payload, key2seq = {}, {}
    for j, (seq, emb) in enumerate(seq2emb.items()):
        k = f'emb_{j:07d}'
        payload[k] = np.asarray(emb, dtype=np.float32)
        key2seq[k] = seq
    cfg.cache_npz.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(cfg.cache_npz, **payload)
    cfg.cache_map.write_text(json.dumps(key2seq))


def load_existing_cache(cfg: CFG) -> dict[str, np.ndarray]:
    if cfg.cache_npz.exists() and cfg.cache_map.exists():
        npz = np.load(cfg.cache_npz, allow_pickle=True)
        key2seq = json.loads(cfg.cache_map.read_text())
        return {seq: np.asarray(npz[k], dtype=np.float32) for k, seq in key2seq.items()}
    return {}


def compute_or_load_cache(df_heavy: pd.DataFrame, cfg: CFG) -> dict[str, np.ndarray]:
    uniq = df_heavy['heavy'].drop_duplicates().tolist()
    seq2emb = load_existing_cache(cfg)

    missing = [s for s in uniq if s not in seq2emb]
    print(f'Cache status: have={len(seq2emb)} need={len(uniq)} missing={len(missing)}')

    if len(missing) == 0 and not cfg.recompute_embeddings:
        return seq2emb

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = transformers.GPT2LMHeadModel.from_pretrained(CHECKPOINT_DICT[cfg.model_name]).to(device)
    model.eval()
    tok2id = load_vocab()

    failures = 0
    for i, seq in enumerate(missing, start=1):
        try:
            seq2emb[seq] = embed_heavy(model, device, tok2id, seq, cfg.species_token)
        except Exception:
            failures += 1
        if i % 50 == 0 or i == len(missing):
            print(f'embedded missing {i}/{len(missing)} (failures={failures})')

    save_cache(seq2emb, cfg)
    print('Saved cache:', cfg.cache_npz)
    return seq2emb


seq2emb = compute_or_load_cache(study_df, cfg)

kept_idx, X_list, y_list = [], [], []
for i, r in study_df.iterrows():
    emb = seq2emb.get(r['heavy'])
    if emb is None:
        continue
    kept_idx.append(i)
    X_list.append(emb)
    y_list.append(float(r['y']))

study_used = study_df.loc[kept_idx].reset_index(drop=True)
X = np.stack(X_list)
y = np.asarray(y_list, dtype=np.float32)

print('Feature matrix:', X.shape, 'Target shape:', y.shape, 'Rows used:', len(study_used))


## 4) Train/validation with MLP only


In [ ]:
# Single train/validation split on the largest study.
idx = np.arange(len(y))
idx_tr, idx_va = train_test_split(idx, test_size=cfg.val_frac, random_state=cfg.seed, shuffle=True)

X_tr, y_tr = X[idx_tr], y[idx_tr]
X_va, y_va = X[idx_va], y[idx_va]
print('Rows train/val:', len(y_tr), len(y_va))

# Train-only standardization.
x_mean = X_tr.mean(axis=0, keepdims=True)
x_std = X_tr.std(axis=0, keepdims=True)
x_std[x_std < 1e-8] = 1.0
y_mean = float(y_tr.mean())
y_std = float(y_tr.std()) if float(y_tr.std()) > 1e-8 else 1.0

X_tr_s = (X_tr - x_mean) / x_std
X_va_s = (X_va - x_mean) / x_std
y_tr_s = (y_tr - y_mean) / y_std
y_va_s = (y_va - y_mean) / y_std

class SmallMLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SmallMLP(X_tr.shape[1], cfg.hidden_dim, cfg.dropout).to(device)

tr_ds = TensorDataset(torch.tensor(X_tr_s, dtype=torch.float32), torch.tensor(y_tr_s, dtype=torch.float32))
tr_dl = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True)
va_x = torch.tensor(X_va_s, dtype=torch.float32, device=device)
va_y = torch.tensor(y_va_s, dtype=torch.float32, device=device)

opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
loss_fn = nn.MSELoss()

best_state = None
best_val = float('inf')
bad_epochs = 0
tr_curve, va_curve = [], []

for _ in range(cfg.max_epochs):
    model.train()
    batch_losses = []
    for xb, yb in tr_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
        batch_losses.append(float(loss.detach().cpu()))

    tr_loss = float(np.mean(batch_losses))
    model.eval()
    with torch.no_grad():
        va_loss = float(loss_fn(model(va_x), va_y).detach().cpu())

    tr_curve.append(tr_loss)
    va_curve.append(va_loss)

    if va_loss < best_val - 1e-6:
        best_val = va_loss
        bad_epochs = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad_epochs += 1
        if bad_epochs >= cfg.patience:
            break

model.load_state_dict(best_state)
with torch.no_grad():
    p_va_s = model(torch.tensor(X_va_s, dtype=torch.float32, device=device)).cpu().numpy()
p_va = p_va_s * y_std + y_mean

mae = float(mean_absolute_error(y_va, p_va))
rmse = float(math.sqrt(mean_squared_error(y_va, p_va)))
r2 = float(r2_score(y_va, p_va))
print(f'Validation MAE={mae:.3f} RMSE={rmse:.3f} R2={r2:.3f} epochs={len(tr_curve)}')


## 5) Validation diagnostics


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.plot(tr_curve, label='train_mse')
plt.plot(va_curve, label='val_mse')
plt.xlabel('epoch')
plt.ylabel('MSE (standardized y)')
plt.title('Train vs Validation Loss')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(5,5))
plt.scatter(y_va, p_va, alpha=0.7)
mn = float(min(y_va.min(), p_va.min()))
mx = float(max(y_va.max(), p_va.max()))
plt.plot([mn, mx], [mn, mx], '--')
plt.xlabel('True Tm (validation)')
plt.ylabel('Predicted Tm (validation)')
plt.title(f'Validation: R2={r2:.3f}, MAE={mae:.2f}')
plt.tight_layout()
plt.show()
